In [1]:
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00


In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoConfig,
    BitsAndBytesConfig
)

In [29]:
# ---------------------------
# 1️⃣ Setup
# ---------------------------

teacher_id = "microsoft/phi-2"
student_id = "microsoft/phi-1_5"

temperature = 2.0
alpha_soft = 0.7
lr = 2e-5
epochs = 3
max_len = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [30]:
# ---------------------------
# 2️⃣ Prompts
# ---------------------------

prompts = [
    "Explain why the sky is blue. ### The sky appears blue because molecules in Earth's atmosphere scatter sunlight, and blue light is scattered more than other colors.",
    "What is the capital of France? ### The capital of France is Paris.",
    "Write a short story about a robot and a cat. ### Once upon a time, a lonely robot found a stray cat. They became best friends, exploring the city together."
]

In [31]:
# ---------------------------
# 3️⃣ Tokenizers
# ---------------------------

teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_id)
student_tokenizer = AutoTokenizer.from_pretrained(student_id)

teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
student_tokenizer.pad_token = student_tokenizer.eos_token

In [32]:
# ---------------------------
# 4️⃣ Load Teacher (4-bit)
# ---------------------------

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

teacher_config = AutoConfig.from_pretrained(teacher_id)
teacher_config.pad_token_id = teacher_config.eos_token_id

teacher = AutoModelForCausalLM.from_pretrained(
    teacher_id,
    config=teacher_config,
    device_map="auto",
    quantization_config=bnb_config
)

teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
# ---------------------------
# 5️⃣ Load Student (Trainable)
# ---------------------------

student_config = AutoConfig.from_pretrained(student_id)
student_config.pad_token_id = student_config.eos_token_id

student = AutoModelForCausalLM.from_pretrained(
    student_id,
    config=student_config,
    torch_dtype=torch.float16
).to(device)

student.train()

In [ ]:
# ---------------------------
# 6️⃣ Optimizer + Loss
# ---------------------------

ce_loss = nn.CrossEntropyLoss(ignore_index=-100)
kl_loss = nn.KLDivLoss(reduction="batchmean")

optimizer = optim.AdamW(student.parameters(), lr=lr)

In [ ]:
# ---------------------------
# 7️⃣ Tokenize Batch
# ---------------------------

t_inputs = teacher_tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=max_len
).to(teacher.device)

s_inputs = student_tokenizer(
    prompts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=max_len
).to(device)

input_ids = s_inputs["input_ids"]
attention_mask = s_inputs["attention_mask"]

labels = input_ids.clone()
labels[attention_mask == 0] = -100  # ignore padding

In [ ]:
# ---------------------------
# 8️⃣ Distillation Training Loop
# ---------------------------

for epoch in range(epochs):

    # Teacher forward
    with torch.no_grad():
        t_outputs = teacher(**t_inputs)
        t_logits = t_outputs.logits

    # Student forward
    s_outputs = student(**s_inputs)
    s_logits = s_outputs.logits

    # Shift for next-token prediction
    t_logits = t_logits[:, :-1, :]
    s_logits = s_logits[:, :-1, :]
    shifted_labels = labels[:, 1:]

    # Soft loss (KL)
    t_probs = torch.softmax(t_logits / temperature, dim=-1)
    s_log_probs = torch.log_softmax(s_logits / temperature, dim=-1)

    loss_soft = kl_loss(
        s_log_probs.reshape(-1, s_log_probs.size(-1)),
        t_probs.reshape(-1, t_probs.size(-1))
    ) * (temperature ** 2)

    # Hard loss (CE)
    loss_hard = ce_loss(
        s_logits.reshape(-1, s_logits.size(-1)),
        shifted_labels.reshape(-1)
    )

    # Combine
    loss = alpha_soft * loss_soft + (1 - alpha_soft) * loss_hard

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1} | Loss: {loss.item():.4f}")

In [ ]:
# ---------------------------
# 9️⃣ Test Generation
# ---------------------------

student.eval()

test_prompt = "Explain why the sky is blue. ###"

inputs = student_tokenizer(test_prompt, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = student.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.7
    )

print("\nGenerated Output:\n")
print(student_tokenizer.decode(outputs[0], skip_special_tokens=True))